## Імпорт необхідних бібліотек

In [12]:
import os
import urllib.request
from datetime import datetime
import pandas as pd
import numpy as np
import timeit

## Створення папки для збереження завантажених файлів

In [13]:
DATA_FOLDER = "data"
os.makedirs(DATA_FOLDER, exist_ok=True)

## Функція завантаження VHI-даних з NOAA

In [14]:
def download_vhi(province_id):
    if province_id == 0:
        print("Province 0 (Ukraine average) is not allowed.")
        return
    
    # перевірка чи вже є файл для цієї області
    existing_files = [
        f for f in os.listdir(DATA_FOLDER)
        if f.startswith(f"VHI_{province_id}_")
    ]
    
    if existing_files:
        print(f"Data for province {province_id} already exists.")
        return
    
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"VHI_{province_id}_{timestamp}.csv"
    filepath = os.path.join(DATA_FOLDER, filename)
    
    urllib.request.urlretrieve(url, filepath)
    print(f"Downloaded {filename}")

## Завантаження даних для всіх областей України

In [15]:
for province in range(1, 28):
    download_vhi(province)

Downloaded VHI_1_20260220_121139.csv
Downloaded VHI_2_20260220_121140.csv
Downloaded VHI_3_20260220_121141.csv
Downloaded VHI_4_20260220_121142.csv
Downloaded VHI_5_20260220_121143.csv
Downloaded VHI_6_20260220_121144.csv
Downloaded VHI_7_20260220_121145.csv
Downloaded VHI_8_20260220_121146.csv
Downloaded VHI_9_20260220_121146.csv
Downloaded VHI_10_20260220_121147.csv
Downloaded VHI_11_20260220_121148.csv
Downloaded VHI_12_20260220_121149.csv
Downloaded VHI_13_20260220_121150.csv
Downloaded VHI_14_20260220_121151.csv
Downloaded VHI_15_20260220_121152.csv
Downloaded VHI_16_20260220_121153.csv
Downloaded VHI_17_20260220_121154.csv
Downloaded VHI_18_20260220_121155.csv
Downloaded VHI_19_20260220_121156.csv
Downloaded VHI_20_20260220_121157.csv
Downloaded VHI_21_20260220_121158.csv
Downloaded VHI_22_20260220_121159.csv
Downloaded VHI_23_20260220_121200.csv
Downloaded VHI_24_20260220_121201.csv
Downloaded VHI_25_20260220_121202.csv
Downloaded VHI_26_20260220_121202.csv
Downloaded VHI_27_202

## Зчитування завантажених файлів у pandas DataFrame

In [25]:
DATA_FOLDER = "data"
all_data = []
    
for file in os.listdir(DATA_FOLDER):
    if file.endswith(".csv"):
        filepath = os.path.join(DATA_FOLDER, file)
            
        df = pd.read_csv(filepath, skiprows=2, header=None)
            
        province_id = int(file.split("_")[1])
        df["province_id"] = province_id
            
        all_data.append(df)
    
df = pd.concat(all_data, ignore_index=True)
df.head(15)

,0,1,2,3,4,5,6,7,province_id
0,<tt><pre>1982,1.0,0.059,258.24,51.11,48.78,49.95,NaN,10
1,1982,2.0,0.063,261.53,55.89,38.20,47.04,NaN,10
2,1982,3.0,0.063,263.45,57.30,32.69,44.99,NaN,10
3,1982,4.0,0.061,265.10,53.96,28.62,41.29,NaN,10
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,NaN,10
5,1982,6.0,0.056,267.47,39.55,30.27,34.91,NaN,10
6,1982,7.0,0.055,268.58,35.19,31.10,33.14,NaN,10
7,1982,8.0,0.057,270.15,33.35,32.09,32.72,NaN,10
8,1982,9.0,0.057,271.60,30.82,34.71,32.77,NaN,10
9,1982,10.0,0.057,273.10,27.66,36.79,32.23,NaN,10


## Data Cleaning
- Видалення зайвих колонок
- Обробка пропусків
- Перейменування колонок

In [29]:
df_clean = df.copy()

# Назви колонок
df_clean.columns = ["year", "week", "SMN", "SMT", "VCI", "TCI", "VHI", "Empty", "province_id"]

df_clean = df_clean.drop(columns=["Empty"])

df_clean['year'] = df_clean['year'].astype(str).str.replace('<tt><pre>', '', regex=False)
df_clean['year'] = df_clean['year'].astype(str).str.replace('</pre></tt>', '', regex=False)

df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce')

df_clean = df_clean.dropna(subset=['year'])
df_clean['year'] = df_clean['year'].astype(int)

df_clean = df_clean[df_clean['VHI'] != -1]

noaa_to_ukr = {
    1: (22, "Черкаська"), 2: (24, "Чернігівська"), 3: (23, "Чернівецька"),
    4: (25, "АР Крим"), 5: (3, "Дніпропетровська"), 6: (4, "Донецька"),
    7: (8, "Івано-Франківська"), 8: (19, "Харківська"), 9: (20, "Херсонська"),
    10: (21, "Хмельницька"), 11: (9, "Київська"), 12: (26, "м. Київ"),
    13: (10, "Кіровоградська"), 14: (11, "Луганська"), 15: (12, "Львівська"),
    16: (13, "Миколаївська"), 17: (14, "Одеська"), 18: (15, "Полтавська"),
    19: (16, "Рівненська"), 20: (27, "м. Севастополь"), 21: (17, "Сумська"),
    22: (18, "Тернопільська"), 23: (6, "Закарпатська"), 24: (1, "Вінницька"),
    25: (2, "Волинська"), 26: (7, "Запорізька"), 27: (5, "Житомирська")
}
df_clean['province_name'] = df_clean['province_id'].map(lambda x: noaa_to_ukr[x][1])
df_clean['province_id'] = df_clean['province_id'].map(lambda x: noaa_to_ukr[x][0])

df_clean = df_clean.reset_index(drop=True)

df_clean.head(15)

,year,week,SMN,SMT,VCI,TCI,VHI,province_id,province_name
0,1982,1.0,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2.0,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
2,1982,3.0,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
3,1982,4.0,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,21,Хмельницька
5,1982,6.0,0.056,267.47,39.55,30.27,34.91,21,Хмельницька
6,1982,7.0,0.055,268.58,35.19,31.10,33.14,21,Хмельницька
7,1982,8.0,0.057,270.15,33.35,32.09,32.72,21,Хмельницька
8,1982,9.0,0.057,271.60,30.82,34.71,32.77,21,Хмельницька
9,1982,10.0,0.057,273.10,27.66,36.79,32.23,21,Хмельницька


## Ряд VHI для області за вказаний рік

In [ ]:
def get_vhi_province_year(df, province_id, year):
    result = df[(df['province_id'] == province_id) & (df['year'] == year)]
    return result[['week', 'VHI']]
# Приклад використання функції для області з ID 22 "Черкаська" та року 2000
vhi = get_vhi_province_year(df_clean, province_id=22, year=2000)
vhi.head(10)


,week,VHI
22766,1.0,35.79
22767,2.0,37.89
22768,3.0,37.46
22769,4.0,36.62
22770,5.0,37.63
22771,6.0,38.49
22772,7.0,36.49
22773,8.0,35.46
22774,9.0,36.99
22775,10.0,38.71


## Ряд VHI за вказаний діапазон років для вказаних областей

In [53]:
def get_vhi_provinces_and_years(df, province_ids, year_min, year_max):
    result = df[
        (df['province_id'].isin(province_ids)) &
        (df['year'].between(year_min, year_max))
    ]
    return result[['province_id', 'year', 'week', 'VHI']]

target_provinces = [6, 22, 25]
vhi_multi = get_vhi_provinces_and_years(df_clean, province_ids=target_provinces, year_min=1995, year_max=1996)
vhi_multi.head(110)


,province_id,year,week,VHI
22509,22,1995,4.0,46.02
22510,22,1995,5.0,40.85
22511,22,1995,6.0,37.51
22512,22,1995,7.0,34.56
22513,22,1995,8.0,32.93
...,...,...,...,...
31257,6,1995,8.0,44.39
31258,6,1995,9.0,45.71
31259,6,1995,10.0,48.40
31260,6,1995,11.0,51.09


## Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [ ]:
def get_vhi_statistics(df, province_ids, year_min, year_max):

    # Фільтрація даних за списком областей та діапазоном років
    filtered_df = df[
        (df['province_id'].isin(province_ids)) &
        (df['year'].between(year_min, year_max))]
    
    # Перевірка, якщо за вказаними параметрами немає даних
    if filtered_df.empty:
        raise RuntimeError("Немає даних для вказаних параметрів.")
    
    # Обчислення статистики для колонки VHI за допомогою .agg()
    stats = filtered_df['VHI'].agg(['min', 'max', 'mean', 'median'])
    
    # Округлення середнього значення
    stats['mean'] = round(stats['mean'], 2)
    
    result_df = pd.DataFrame(stats).T
    result_df.index = ['VHI Stats']
    
    return result_df

# Приклад виклику функції: шукаємо статистику для Вінницької (1) та Київської (9) областей за 2010-2015 роки
target_provinces = [1, 9]
vhi_stats = get_vhi_statistics(df_clean, province_ids=target_provinces, year_min=2010, year_max=2015)

print(f"Статистика VHI для областей {target_provinces} за період з 2010 по 2015 рік:")
display(vhi_stats)

Статистика VHI для областей [1, 9] за період з 2010 по 2015 рік:


,min,max,mean,median
VHI Stats,19.94,74.38,46.19,45.15
